# 04 · 消融与专项矩阵

核心入口仍是 `run_matrix.py`。默认运行 A0–A6、seed 0；先稳定到 `[0, 1, 2]`，最终再统一到 `[0, 1, 2, 3, 4]`。可通过一个参数选择多尺度、时间、校准、backbone、dataset view 或 efficiency 正式配置组。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"
DATASET = "THEIA_E3"
SEEDS = [0]  # 建议先 [0, 1, 2]；稳定后正式使用 [0, 1, 2, 3, 4]
EXPERIMENT_GROUP = "ablation"
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)
assert DATASET in {"THEIA_E3", "THEIA_E5"}

In [ ]:
GROUPS = {
    "ablation": [
        "baseline.yml", "ablation_no_multiscale.yml", "ablation_no_gate.yml",
        "ablation_no_time.yml", "ablation_no_calibration.yml",
        "ablation_no_topk.yml", "mstc_full.yml",
    ],
    "multiscale": [
        "multiscale_recent20.yml", "multiscale_recent24.yml",
        "multiscale_single_window.yml", "multiscale_equal.yml", "multiscale_gate.yml",
    ],
    "time": ["time_type_only.yml", "time_time_only.yml", "time_joint.yml"],
    "calibration": [
        "calibration_max.yml", "calibration_quantile.yml", "calibration_kmeans.yml",
        "calibration_global_p.yml", "calibration_relation.yml", "calibration_hierarchical.yml",
    ],
    "backbone": [
        "backbone_graphtransformer.yml", "backbone_graphsage_baseline.yml",
        "backbone_graphsage.yml", "backbone_mlp.yml",
    ],
    "dataset_view": ["host_only.yml", "host_network_structure.yml", "host_network_full.yml"],
    "efficiency": [
        "baseline.yml", "efficiency_multiscale.yml",
        "efficiency_multiscale_time.yml", "mstc_full.yml",
    ],
}
if EXPERIMENT_GROUP not in GROUPS:
    raise ValueError(f"未知组 {EXPERIMENT_GROUP!r}；可选 {sorted(GROUPS)}")
CONFIGS = [PROJECT_ROOT / "config/experiments" / name for name in GROUPS[EXPERIMENT_GROUP]]
missing = [path for path in CONFIGS if not path.is_file()]
if missing:
    raise FileNotFoundError(missing)
print("配置组：", EXPERIMENT_GROUP)
for path in CONFIGS:
    print(" -", path.name)

## OOM 公平性规则

OOM 时当前 run 记为 failed，matrix runner 继续并保留 failure metadata。不要由 Notebook 自动降低 batch size、candidate capacity、neighbor budget 或 hidden dimension；causal micro-batch 下 batch size 也可能改变上下文。应人工制定一份统一配置，并对所有公平对照重新运行。

In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "src/experiments/run_matrix.py"),
    "--datasets", DATASET,
    "--configs", ",".join(map(str, CONFIGS)),
    "--seeds", ",".join(map(str, SEEDS)),
    "--artifact-root", str(ARTIFACT_ROOT),
]
result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)
print("run_matrix exit code:", result.returncode)
if result.returncode:
    raise subprocess.CalledProcessError(result.returncode, command)